In [ ]:
import os
import sys
import subprocess
from lucifex.io import create_dir_path
from lucifex.sim.parallel import combine_options
from crocodil.dns.system_a import dns_system_a, SYSTEM_A_REFERENCE

sr_opts = ()
Ra_ref = SYSTEM_A_REFERENCE['Ra']
Ra_opts = tuple(i * Ra_ref for i in (1.0, 0.5, 2.0))
Da_ref = SYSTEM_A_REFERENCE['Da']
Da_opts = tuple(i * Da_ref for i in (1.0, 0.1, 10.0))

SCRIPT = 'run_system_a.py'
OVERWRITE = False

CONFIG_PARAMS = dict(
    store_delta=None,
    write_delta=0.01,
    dir_root='./data',
    dir_params=('Ra', 'Da', 'sr'),
    dir_uid=True,
)
SIMULATION_PARAMS = dict(
    Nx=200,
    Ny=200,
    courant_adv=0.75,
    courant_diff=0.75,
    courant_reac=0.1,
    c_limits=True,
)
RUN_PARAMS = dict(
    n_stop=9999,
    t_stop=120.0,
)

opt_names = ('Ra', 'Da', 'sr')
run_opts = []
for Ra, Da, sr in combine_options(sr_opts, Ra_opts, Da_opts, link=False):
    simulation_params = dict(
        Ra=Ra,
        Da=Da,
        sr=sr,
        **SIMULATION_PARAMS,
    )
    dir_path = create_dir_path(
        simulation_params,
        **CONFIG_PARAMS
    )
    if not OVERWRITE and not os.path.exists(dir_path):
        print(f'Scheduled to run {opt_names} = {Ra, Da, sr}')
        run_opts.append((Ra, Da, sr))
    else:
        print(f'Already run {opt_names} = {Ra, Da, sr}')

In [ ]:
log = open(f"{SCRIPT}.log", "a")
for Ra, Da, sr in run_opts:
    cli_dict = {
        **CONFIG_PARAMS,
        **SIMULATION_PARAMS,
        **dict(zip(('Ra', 'Da', 'sr'), (Ra, Da, sr))),
        **RUN_PARAMS,
    }
    cli_kwargs = [(f'--{k}', str(v)) for k, v in cli_dict.items()]
    p = subprocess.Popen(
        [
            "nohup", sys.executable, SCRIPT, 
            "--Ra", str(Ra),
            "--Da", str(Da),
            "--sr", str(sr),
            *[i for pair in cli_kwargs for i in pair]
        ],
        stdout=log, 
        stderr=log,
        preexec_fn=os.setsid,
    )
    print(f"(Ra, Da, sr) = {Ra, Da, sr} PID:", p.pid)

PID: 31309
